# World

Is the infinite world (`docs/plans/infinite-plan.md`) right? This notebook answers in five parts:

1. **Each level against the simulator, on inputs like its training data.** Did the models learn their
   task?
2. **The world against the simulator on the same terrain.** Is the world's output right? This is the
   main test.
3. **A large region.** What the world looks like.
4. **Seams.** Do neighboring windows agree before they are blended?
5. **Machinery.** Query order, the cache, and speed.

Needs the two world checkpoints (`artifacts/checkpoints/diffusion-world_coarse.pt`,
`diffusion-world_fine.pt`; profiles `world_coarse` and `world_fine` in `notebooks/chunk/train.ipynb`).
Part 2 also needs the `world_master` dataset (generate it in `notebooks/world/data.ipynb`); it uses one
of its large test samples.

To explore the world in a browser instead: `uv run python -m erosion_sim.world.viewer`, then open the
address it prints.

**Terms.** `Δz` is the change in height (m), end minus start; negative is erosion. RMSE is the root
mean square error of the end heights against the simulator (m), lower is better. Skill is
`1 - MSE / MSE(no change)`: 0 is no better than leaving the terrain as it is, 1 is perfect. Channel IoU
(intersection over union) compares where rivers are, within one cell: 1 is the same rivers as the
simulator. Pits are cells lower than all their neighbors; water stops there, so each one breaks a river.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from erosion_sim.evaluate import spectrum_comparison
from erosion_sim.render import hillshade
from erosion_sim.data import Split
from erosion_sim.world.compare import Version, compare_versions
from erosion_sim.world.evaluate import (by_elapsed, chunked_vs_whole, edge_profile, headline, level_pairs,
                                        order_invariance, query_times, run_world, seam_summary, show_examples,
                                        simulator_times, speed_comparison, window_disagreement)
from erosion_sim.world.level import SPLIT
from erosion_sim.world.world import World, load_levels

LEVELS = load_levels()  # diffusion-world_coarse.pt and diffusion-world_fine.pt

# The world used in parts 3 to 5. Parameters are mid-range of training.
SEED, DT = 1, 30_000.0
PARAMS = dict(K_ref=2.5e-4, D=2.5e-4, m=0.45, persistence=0.5)


def make_world(dT=DT, splits=(SPLIT,), **kw):
    return World(SEED, dT, *LEVELS, splits=splits, **PARAMS, **kw)

## 1. Each level against the simulator, on training-like inputs

Validation runs (128² cells, no uplift, every elapsed time), the same kind of data the models trained on.
This tests the models, not the world: each run here is a closed domain with fixed edges.

- **Coarse level:** whole runs as 4×4 block means (32² cells of 4 m).
- **Fine level:** 64 m crops at 1 m. The fine model reads the 1 m starting terrain and a coarse map of
  `Δz` at 4 m cells. That map comes from the simulator (its true change, block-averaged; only known in
  testing, and what training used) or from the coarse model (what the world uses).

| Method (fine level) | What it is | Compare it with |
|---|---|---|
| **coarse + fine models** | the coarse model's map, then the fine model: **the world's pipeline** | everything below |
| coarse model only | the coarse model's map, smoothly upsampled; no fine model | the line above: what the fine level adds |
| fine model, simulator's coarse | the fine model given the true map | the first line: what coarse errors cost |
| simulator's coarse only | the true map, smoothly upsampled; no fine model | the line above: what the fine model adds at best |
| no change | the starting terrain | the floor: every method should beat it |

In [ ]:
PAIRS = level_pairs(*LEVELS, count=256)
print("COARSE LEVEL")
coarse_table = headline(PAIRS["coarse"])
print("\nFINE LEVEL")
fine_table = headline(PAIRS["fine"])

**How to read the maps below.** For each example, the top row is the starting terrain, then `Δz` of the
simulator and of each method, all on one color scale. The bottom row is each method's error
(method minus simulator), on its own scale. Good methods have a top row that looks like the
simulator's and a pale bottom row.

In [ ]:
show_examples(PAIRS["coarse"], ["coarse model"], title="Coarse level: whole runs at 4 m cells")
FINE_METHODS = ["coarse + fine models", "coarse model only", "fine model, simulator's coarse"]
show_examples(PAIRS["fine"], FINE_METHODS, title="Fine level: 64 m crops at 1 m cells")

**Against elapsed time.** Longer elapsed times have more change, so all errors grow; "no change" (dashed)
grows fastest. Channel IoU falls with time for every method, since rivers move further from where
the starting terrain would put them.

In [ ]:
elapsed = by_elapsed(PAIRS["fine"], FINE_METHODS)

**Spectrum.** How much change there is at each size of feature, as a ratio to the simulator (1 is
right). The baselines without a fine model fall off at short wavelengths: they have no fine detail.
A method whose ratio stays near 1 at short wavelengths adds detail of the right size.

In [ ]:
spectrum = spectrum_comparison(dict(PAIRS["fine"], methods=["no change", *FINE_METHODS, "simulator's coarse only"]))

## 2. The world against the simulator on the same terrain

**The question:** if you ask the world for one chunk at a time, each with no knowledge of the others,
and tile the chunks together, do you get what the simulator gives when it runs across the whole area at
once?

The world's starting terrain exists everywhere, so it can be built on the same terrain as a simulator
run. One 512² test sample of `world_master` is the answer key; the world uses that sample's terrain and
erosion parameters. Only its interior is used, 64 cells in from its edges, since the sample's own edges
are fixed and drain water out, while the world has no edges. The scores come from `compare_versions`,
the same grading `compare.ipynb` uses for many samples.

| Method | What it is |
|---|---|
| **world** | the region asked of one world in one query |
| world, coarse level only | the world's coarse `Δz`, smoothly upsampled, no fine level |
| no change | the starting terrain |

Then the same region is asked for in 128² chunks, each from a brand-new world (empty cache), and tiled
together. If chunks are truly independent, that equals the one query up to float rounding. Black lines in
the maps are those chunk edges; seams would line up with them.

The large run is not a perfect stand-in for an edgeless world: its water still leaves through its own
edges eventually.

In [ ]:
REFERENCE = Split.open("world_master", "test")
SAMPLE = 0  # which test sample is the answer key
print(REFERENCE.terrain(SAMPLE), REFERENCE.erosion(SAMPLE))
WORLD_VS_SIM = compare_versions({"world": Version("world_coarse", "world_fine")}, runs=[SAMPLE], coarse_only=True)
CHUNKED = chunked_vs_whole(lambda: run_world(REFERENCE, SAMPLE, 30_000.0, *LEVELS), 64, 64, 384, chunk=128)

In [ ]:
show_examples(WORLD_VS_SIM["pairs"], ["world", "world, coarse level only"], examples=(0, 1, 2),
              grid=CHUNKED["grid"], simulator="large simulator run",
              title="Interior 384² (1 m cells) of one 512² sample. Black lines: 128² chunk edges.")

## 3. A large region

1024² cells (1 km) of the world at `dT` = 30,000 yr, with T = 2 (the sampler's schedule split at
σ = 5 σ_data, windows blended in between) and T = 1 (windows blended only at the end). No answer key
here: this shows what the world looks like, not whether it is right.

In [ ]:
N = 1024
worlds = {"T = 2": make_world(), "T = 1": make_world(splits=())}
heights = {name: w.get(0, 0, N, N) for name, w in worlds.items()}
start = worlds["T = 2"].terrain.heights(0, 0, N, N)
change = heights["T = 2"] - start

fig, ax = plt.subplots(2, 2, figsize=(14, 14), layout="constrained")
ax[0, 0].imshow(hillshade(start)); ax[0, 0].set_title("starting terrain")
ax[0, 1].imshow(hillshade(heights["T = 2"])); ax[0, 1].set_title(f"eroded, T = 2, dT {DT:.0f} yr")
lim = np.abs(change).max()
image = ax[1, 0].imshow(change, cmap="RdBu_r", vmin=-lim, vmax=lim); ax[1, 0].set_title("Δz, T = 2")
fig.colorbar(image, ax=ax[1, 0], label="Δz (m)", shrink=0.8)
d = heights["T = 2"] - heights["T = 1"]
dlim = np.abs(d).max()
image = ax[1, 1].imshow(d, cmap="RdBu_r", vmin=-dlim, vmax=dlim)
ax[1, 1].set_title(f"T = 2 minus T = 1 (RMS {np.sqrt((d ** 2).mean()):.2f} m)")
fig.colorbar(image, ax=ax[1, 1], label="m", shrink=0.8)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.show()

In [ ]:
# A 128 m zoom. Thin lines are the fine windows' stride (32 m); a seam would line up with them.
zoom = (slice(448, 576), slice(448, 576))
fig, ax = plt.subplots(1, 3, figsize=(18, 6), layout="constrained")
ax[0].imshow(hillshade(start[zoom])); ax[0].set_title("start, 128 m")
for a, (name, z) in zip(ax[1:], heights.items()):
    a.imshow(hillshade(z[zoom])); a.set_title(f"eroded, {name}")
    for k in range(0, 129, 32):
        a.axhline(k - 0.5, color="k", lw=0.4, alpha=0.3); a.axvline(k - 0.5, color="k", lw=0.4, alpha=0.3)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.show()

## 4. Seams

Each window is sampled on its own, then overlapping windows are averaged. `window_disagreement`
compares each window's own output with its neighbors' where they overlap, before that average.

The reference is each level's own error against the simulator from part 1 (gray). If neighboring
windows disagree by more than the model errs, the average mixes different answers: the world is less
certain than any one window, and the fine level inherits that. Disagreement well below the model's
error is what we want.

In [ ]:
seams = {}
for name, splits in [("T = 1", ()), ("T = 2", (SPLIT,))]:
    w = make_world(splits=splits)
    seams[name] = {level: window_disagreement(getattr(w, level), show=False) for level in ("coarse", "fine")}
own_error = {"coarse": coarse_table["coarse model"]["rmse"], "fine": fine_table["fine model, simulator's coarse"]["rmse"]}
seam_result = seam_summary(seams, own_error)
for name, levels in seam_result.items():
    print(name, {level: f"{rms:.2f} m" for level, rms in levels.items()})

**Change near window edges.** Each window's mean `|Δz|` against distance from its own edge. The
simulator's edges never change, so a model trained on whole runs learns to erode less near any edge,
and does so at every window edge in the world. Only the outermost cells matter much here, since the blend
gives them almost no weight.

In [ ]:
w = make_world()
profiles = {level: edge_profile(getattr(w, level), show=False) for level in ("coarse", "fine")}
fig, ax = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for a, (level, p) in zip(ax, profiles.items()):
    a.plot(p["distance"], p["mean_abs_change"], marker="o", ms=3)
    a.set_xlabel(f"distance from the window edge ({level} cells)"); a.set_ylabel("mean |Δz| (m)")
    a.set_title(f"{level} windows"); a.set_ylim(0, None)
plt.show()

## 5. Machinery

**Query order and the cache.** Two fresh worlds answer the same random regions in different orders; one
clears its cache halfway. The answers can differ only by float rounding: windows are summed and
batched in different orders.

In [ ]:
invariance = order_invariance(make_world, regions=12, size=96)

**Speed against the simulator.** Seconds to get one square region of eroded terrain.

- **Simulator:** one run over the region, on one CPU core (as in data generation), with the
  world's terrain and parameters. Its time grows with the elapsed time: one step per 100 yr.
- **World, first query:** a fresh world with an empty cache. It computes every coarse and fine window the
  region touches, through both phases. Its time does not depend on the elapsed time.
- **World, cached:** the same query again, read from the cache.

Two world settings: the notebook's (32 sampler steps, float32) and the viewer's (16 steps, bfloat16).
The simulator could also run separate chunks on many cores at once, but fixed chunk edges change the
answer. The world gives one coherent answer from separate queries.


In [ ]:
times = {
    "32 steps, fp32": query_times(make_world, sizes=(64, 128, 256, 512, 1024), show=False),
    "16 steps, bf16": query_times(lambda: make_world(steps=16, bf16=True), sizes=(64, 128, 256, 512, 1024), show=False),
}
simulator = simulator_times(make_world(), sizes=(64, 128, 256, 512, 1024))
speed = speed_comparison(times, simulator)